# New horizontal integer unpacking algorithm for Apache Parquet

## Integer packing

### Integer representation

Integer bit packing is the operation of writing an integer on a smaller number of bits than what is required to work with it on a computer.
Today most computer must work with multiple of 8 bits, also called a byte, and integers are represented with a certain number of bytes.
Most performant reprensentation have a capacity of either 32 or 64 bits.
For the sake of simplicity, let us assume that in these bits, non negative integers are stored in base 2, in increasing base power, and paded to zero for higher unused bits.

For instance the value $13$ can be represented as the following, where contrary to mathematical representation we go from $2^0$ (low order bit) to $2^3$ (high order bit) instead of the opposite, as it will make it easier to work with.
$$13 = 1*2^0 + 0*2^1 + 1*2^2 + 1*2^3 = \underline{1011}_2$$

The following interaction let you explore reprensentation from any number, written over 32 bits.
The green bits are ones with the actual data while the non-colored ones are zero padding to 32 bits.
Padding bits are still valid in the representation, similar to how a number such $13$ can also be written $013$, $0013$, *etc.*
Vertical bars separate bytes from one another.

In [1]:
#include "./include/config.hpp"

In [2]:
#include "all.hpp"

In [3]:
auto uint_wt = IntegerToolbar();

auto uint_on_change = [](auto value, Uint uint) {
    auto const unpacked_bit_size = static_cast<std::size_t>(uint);
    
    std::cout << "Binary integer representation:\n";
    auto const bit_style = MakeUnpackedColorFn({
        .packed_bit_size = static_cast<std::size_t>(std::bit_width(value)),
        .unpacked_bit_size = unpacked_bit_size,
        .bit_highlight = [](auto) { return false; },
    });
  
    const auto value_le = arrow::bit_util::ToLittleEndian(value);
    PrintBytes(value_le, { .bit_style = bit_style, .max_bytes = unpacked_bit_size / 8 });
};

uint_wt.on_change(uint_on_change);
uint_wt.display();

A Jupyter widget with unique id: 1f3f95e9b501474b91c253b24a29b2ad

A Jupyter widget with unique id: d9009be25fb441d29d79a242a455dce5

### Packing a sequence of integers
We can observe that for small numbers, there are few green bits and a lot of redundant zeros.
In general this is not an issue, after all four million such number is roughly 4mB or memory (RAM) used.
Most computers today have at least a thousand times that available.
On top of this, this is simply the minimal representation needed when we need to make something useful with an integer such as an addition, mulitiplication *etc* (while 8 and 16 bit integer are available, applications typically use 32 or 64 bit integers).

The constraints change when we focus on data storage.
Computers can store on disk (SSD) and in the cloud (someone else's disk) much more data than can fit in memory and only access the relevant parts.
There the data stay dormant taking space for as long as it is stored (indefinitely for businesses) so the question of storing only the green part becomes relevant.
In cloud and web environments, moving data over internet is also often the bottleneck.
In this case, reducing the size of the data to transmit is a great speedup.

In practice, this does not happen for any single integer, but rather, for a sequence of integers, such as a column in a database or in a tabular file format such as [Apache Parquet](https://parquet.apache.org/).

Let us see how that would work.
First if we stored the relevant bits for each value in the sequence, all value may take a different size in bit, and we would no longer be able to tell them apart.\
So instead the first step is to find the largest size needed across all our values.
The following function `MaxBitWidth` does just that.

In [4]:
template <typename Range>
auto MaxBitWidth(Range&& in) -> std::size_t {
    auto bit_width = [](auto x) { return std::bit_width(x); };
    const auto value = std::ranges::max(in, {}, bit_width);
    return static_cast<std::size_t>(bit_width(value));
}

Then, we will pack all our values across the maximum number of bits, setting unused higher order bits to zero as previously done.
The following `PackExact` is a naive implementation of such a routine that writes the bits one by one to the output.
Recall that the minimum entity we can address (work with) is a byte (8 bits).
So we use a byte (`std::uint8_t` here) to contain it, shift operation (`<<` and `>>`) to move the data inside the byte, AND operations (`&`) with a mask to isolate the relevant bit, and OR operations (`|`) to set in back in a given byte.

In [5]:
/// Naively pack integers into bit-packed format.
template <typename Range>
  requires std::ranges::input_range<Range>
auto PackExact(Range&& in, int packed_bit_width) -> std::vector<std::uint8_t> {
  const int batch_size = static_cast<int>(in.size());

  // Calculate required output size in bytes
  const int output_bytes = arrow::bit_util::BytesForBits(batch_size * packed_bit_width);
  auto out = std::vector<std::uint8_t>(output_bytes, std::uint8_t{0});

  // Pack each value bit by bit
  int out_bit_idx = 0;
  for (const auto val : in) {
    // Write each bit of this value
    for (int bit_idx = 0; bit_idx < packed_bit_width; bit_idx++) {
      const int out_byte_idx = out_bit_idx / 8;
      const int out_bit_offset_in_byte = out_bit_idx % 8;

      // Extract bit in position bit_idx and move it to the first bit
      constexpr std::uint8_t kFirstBitMask = 0b00000001;
      const std::uint8_t bit = (val >> bit_idx) & kFirstBitMask;
      // Write bit in output byte at correct position
      out[out_byte_idx] |= (bit << out_bit_offset_in_byte);

      out_bit_idx++;
    }
  }

  return out;
}

The following interaction shows the initial values (unpacked) and bit-packed values for a given sequence.

In [6]:
auto packing_wt = SequenceToolbar();

auto packing_on_change = []<typename Widget>(Widget const& w){
    using value_type = typename Widget::value_type;
    auto const values = w.values();

    const auto unpacked_bit_size = static_cast<std::size_t>(w.unpacked_uint_max_bits());
    const auto packed_bit_size = static_cast<std::size_t>(w.packed_bit_size());
    const auto bytes = PackExact(values, packed_bit_size);
    const auto current_val = w.step() % values.size();

    std::cout << "Storing " << values.size() << " unpacked values over "
        << unpacked_bit_size << " bits each takes a total of "
        << sizeof(value_type) * values.size() << " bytes:\n";

    // Abusing PackExact to dynamically static cast to the input unpacked type
    const auto uint_bytes = PackExact(values, unpacked_bit_size);
    PrintBytes(uint_bytes, {
        .lane_bit_size = unpacked_bit_size,
        .lane_end = "\n",
        .bit_style = MakeUnpackedColorFn({
            .packed_bit_size = packed_bit_size,
            .unpacked_bit_size = unpacked_bit_size,
            .selected_value_idx = current_val,
        }),
    });

    std::cout << "\nStoring the same " << values.size()
        << " values packed in " << packed_bit_size << " bits each takes a total of "
        << bytes.size() << " bytes:\n";
    
    PrintBytes(bytes, {
        .lane_bit_size = unpacked_bit_size,
        .lane_end = "\n",
        .bit_style = MakePackedColorFn({
            .packed_bit_size = packed_bit_size,
            .selected_value_idx = current_val,
            .n_valid_bits = values.size() * packed_bit_size,
        }),
    });
};

packing_wt.on_change(std::move(packing_on_change));
packing_wt.display();

A Jupyter widget with unique id: 6c52a1ef6f8f42f6ad921537c24aa030

A Jupyter widget with unique id: f33d5016bf9d45c39609d0b90c67eaaf

A Jupyter widget with unique id: e6c8628c75924951aa86c5d827df89a9

As we can see, for a sequence of small values we can save a lot of space.
How relevant is this?
Well a lot of numbers used by human are generally rather small: number of children, price (in cents) *etc*.
Categories are also encoded as small integer.
In Parquet files this is part of *dictionnary encoding*.
For example, let us imagine a user choice between three frequency of notifications.
This can be represented as:
$$None \rightarrow 0$$
$$Essentials \rightarrow 1$$
$$All \rightarrow 2$$

In Parquet files, column with list types are are encoded using the list indices.
The column following column stored as two separate ones:

| basket                         |
|--------------------------------|
| ["Apple", "Banana"]            |
| ["Banana", "Cookies", "Juice"] |

Becomes

| basket.value | basket.repetition |
|--------------|-------------------|
| "Apple"      | 0                 |
| "Banana"     | 1                 |
| "Banana"     | 0                 |
| "Cookies"    | 1                 |
| "Juice"      | 1                 |

The general schema, called repetition and definition level, is from
Google's [Dremel publication](https://static.googleusercontent.com/media/research.google.com/en//pubs/archive/36632.pdf)
and is beyond the scope of this blog post.
However this is another example of why encoding small numbers appear quite often in Parquet file, even if they are not directly the values that we think of when looking at a column.

We can also notice that a single large value could completely reduce the space saving by forcing all values to be written in over a large number of mostly zero bits.
In practice the packing is done in *runs*, for instance packing 1024 values at a time with the smallest possible bit width.

Packing integers is a tradeoff.
With packed intgers, reading a file requires to unpack them, which is much more costly in computation than simply copying them to the relevant integer type if they were written over 32 or 64 bits.
Instead, we now have to shift and mask byte around, handle values split across bytes, memory cache lines *etc*.

Data is often read more than it is written.
In particular, large Parquet datasets are queried repeatedly by business analysts, data scientists, and maching learning pipelines in modern
[data lakehouses](https://www.databricks.com/glossary/data-lakehouse).
For this reason, the focus of this blog post and of our Apache Parquet contribution is on speeding up the integer unpacking routine to increase reading spead.

## Exact integer unpacking 

The `PackExact` function we used earlier is really aweful in term of performance.
It writes the bits one by one, and every time it needs to make a few  manipulations.
As we now now, the smallest unit we can manipulate is a whole byte, for that reason every time we are manipulating a bit
we are moving around seven other bits for nothing.

Back to our packing figure, we would really want to move these blue and green blocks (that is the actual packed values) in full.
Read the packed value one by one in an integer of the size we want to target and set the unwanted bits to zero.
However, there is an issue: in the same way that we cannot read less than a byte, we must also read it on a multiple of 8 bits.
In other words, the bytes are predefined blocks and we get to pick which block we want.
They are not a sliding window into a memory of bits.
So we need to read a certain number of bytes that will fully contain the current value we want to extract.
But the value will not start on the first bit, so we need to shift it to make it happen.
It will also contains other bits from the next value that we need to erase.
We are doing so by masking with the known number of relevant bits.
The following animation illustrate the algorithm.

In [7]:
auto unpacking_wt = SequenceToolbar({.interval_ms = 1000});

auto unpacking_on_change = []<typename Widget>(Widget const& w){
    using value_type = typename Widget::value_type;
    const auto values = w.values();
    const auto packed_bit_size = static_cast<std::size_t>(w.packed_bit_size());

    const auto params = UnpackExactSelectedParams{
        .value_idx = w.step() % values.size(),
        .n_values = values.size(),
        .packed_bit_size = packed_bit_size,
        .unpacked_bit_size = static_cast<std::size_t>(w.unpacked_uint_max_bits()),
        .max_spread_bytes = static_cast<std::size_t>(arrow::internal::PackedMaxSpreadBytes(packed_bit_size, 0)),
    };
    const auto bytes = PackExact(values, params.packed_bit_size);
    // Reproducing state of previous step
    auto out_values = std::decay_t<decltype(values)>(values.size(), value_type{0});
    std::copy(values.cbegin(), values.cbegin() + params.value_idx, out_values.begin());

    std::cout << "Input packed values:\n";
    PrintBytes(bytes, {
        .bit_style = MakePackedColorFn({
            .packed_bit_size = params.packed_bit_size,
            .selected_value_idx = params.value_idx,
            .n_valid_bits = values.size() * params.packed_bit_size,
        }),
    });

    std::cout << "\nSelect memory region:\n    ";
    PrintPackedSelect(bytes, params);

    auto buffer = std::uint64_t{};
    auto n_bytes_to_copy = std::min(
        params.max_spread_bytes,
        bytes.size() - params.ValueByteStart()
    );
    std::memcpy(&buffer, bytes.data() + params.ValueByteStart(), n_bytes_to_copy);
    buffer = arrow::bit_util::FromLittleEndian(buffer);
    std::cout << "\nCopy the current value and extra bits:\n"
        << "    read(start=" << params.ValueByteStart() << " ,size="
        << n_bytes_to_copy <<")\n    ──► ";
    PrintBufferSelect(buffer, params);

    buffer >>= params.SpreadShiftBits();
    std::cout << "\nShift buffer to align current value:\n    ";
    PrintBufferSelect(buffer, params);  
    std::cout << " << " << params.SpreadShiftBits() << "\n    ──► ";
    PrintShiftedBuffer(buffer, params);
   
    auto const mask = arrow::bit_util::LeastSignificantBitMask<std::uint64_t>(params.packed_bit_size);
    buffer = buffer & mask;
    std::cout << "\nApply bit mask:\n        ";
    PrintShiftedBuffer(buffer, params);
    std::cout << "\n     &  ";
    PrintMask(mask, params);    
    std::cout << "\n    ──► ";
    PrintUnpackedSelect(buffer, params);
    
    out_values.at(params.value_idx) = buffer;
    std::cout << "\nSave to ouput:\n";
    // Abusing PackExact to dynamically static cast to the input unpacked type
    const auto uint_bytes = PackExact(out_values, params.unpacked_bit_size);
    PrintBytes(uint_bytes, {
        .lane_bit_size = params.unpacked_bit_size,
        .lane_end = "\n",
        .bit_style = MakeUnpackedColorFn({
            .packed_bit_size = params.packed_bit_size,
            .unpacked_bit_size = params.unpacked_bit_size,
            .selected_value_idx = params.value_idx,
            .n_colored_values = params.value_idx + 1,
        }),
    });
};

unpacking_wt.on_change(std::move(unpacking_on_change));
unpacking_wt.display();

A Jupyter widget with unique id: 4320e74818f941bbaa02fa763bbd8a95

A Jupyter widget with unique id: 563bb4619cb34d7cb437764ba7825a69

A Jupyter widget with unique id: 36fca38347cc4a7bacb7fa200895f40b

If you were careful, something subtle happens for certain packing bit sizes.
If you look at the algorithm with a packed bit size of 2 or 4, you would notice that we read one byte at a time to extract the value.
Similarily if the packed bit size is 10 or 12, we that we read two bytes at a time.
This seems pretty normal, it is the smalest number of bytes that can contain the given number or bits,
or $\left\lceil{\frac{bit{\_}size}{8}}\right\rceil$.

However, that is not what happens with when the bit size is 3, 5, 6, 7, 11 *etc*.
What happens there is although a 3 bit can fit in a byte, when we pack them together, some value eventually becomes poorly
aligned (the third one in this case) and spread over 2 bytes.
So in this case, we need to extract $\left\lceil{\frac{bit{\_}size}{8}}\right\rceil + 1$ bytes from memory to have the value in full.
This is expressed with the function `PackedMaxSpreadBytes` from our contribution and is at the core of dimensioning our unpacking algorithm.
For instance, with a packing size of 59 bits, the spread becomes of 9 bytes.
That is larger than the largest unsigned integer type available to us (`uint64_t` is 8 bytes).
We call this case *large* and need to handle it appropriately.

This algorithm is implemented in our contribution as the `unpack_exact` function (found in `cpp/src/arrow/util/bpacking_dispatch_internal.h`).
The function is more complex to for reusability.
There are certain differences of interest to us.
First it is *exact*.
If it will unpack exactly the number of values that we ask for, and has the possiblity to resume with a first value non-aligned on a byte.
For example if we extract two 3 bit values from an input, the next one will start at bit 6, which is not a begining of a byte.
Our fuction need to handle such cases to be general.
Second, we should notice that both the packing bit size and the unpacking integer type are taken as template parameters (that is, known at compile time).
Knowing informatio at compile time enable a lot of small optimization, for instance, we can adapt the size of our unsugned integer reading buffer (types need to be decided at compile time in C++).
We can also hard code certain values, such as the bit mask.
These values are transparent to the compiler which can use them to further optimize our function.
However, in practice the packing bit width is only known at runtime, for instance part the encoding in Parquet.
So how should manage this?
Well, since there are only so many possible packing bit widths (up to 64 for `uint64_t`), we compile our function for each of them, and then select the correct one (*jump* to the correct function).
This is achieved by the function `unpack_jump` which does exactly this (albeit a bit verbosly).
This is a common pattern in high performance scientific computing that is know as *microkernel dispatching*.

Was all this necessary?
For this `unpack_exact` this likely does not make huge performance difference, but this function is not reason why it was put in place in Apache Parquet.
It is used as a helper around a function that extract more than one value per iteration.
To get an intuition of how we might do this, look again at the algorithm for packing bit width of say 2.
In this case, we are reading one byte at the time.
But our unsigned integers go up to a size of 8 bytes.
Surely if we read 8 bytes, there could be a way to extract subsequent values more easily.
That is the focus of our contribution to Apache Parquet.
More precisely, it uses dedicated [SIMD](https://en.wikipedia.org/wiki/Single_instruction,_multiple_data) CPU instructions to operate on multiple values at once, in roughly the same time it would take for a single value.
This works because these instructions map to specialized hardware pathways in the CPU.

## Planning at compile time

### On SIMD

SIMD intructions operate on registers of a certain size.
They are available depending on CPU architecture and optional "features" (micro-architecture).
For instance on [x86](https://en.wikipedia.org/wiki/X86) (most computers except recent Apple M-series) CPUs,
128 bits register are introduced with the [SSE](https://en.wikipedia.org/wiki/Streaming_SIMD_Extensions),
and 256 bits registers with [AVX](https://en.wikipedia.org/wiki/Advanced_Vector_Extensions).

They can be used with dedicated types and functions.
[Arm](https://en.wikipedia.org/wiki/ARM_architecture_family) CPU with [Neon](https://en.wikipedia.org/wiki/ARM_architecture_family#Advanced_SIMD_(Neon)) probably have the most easily understandable names.
A `uint32x4_t` is a small array of four `std::uint32_t`.
Two such arrays can be summed with the `vaddvq_u32` function.
The difference compared to doing similar operations on a `std::array<std::uint64_t, 4>` is that the CPU is doing it in a single shot instead of looping.

Using SIMD and microkernel dispatching for unpacking integers is not new.
Our algorithm is based on work from Daniel Lemire and Leonid Boytsov,
[*Decoding billions of integers per second through vectorization*](http://arxiv.org/abs/1209.2137),
and made available the [LittleIntPacker library](https://github.com/fast-pack/LittleIntPacker/blob/master/src/horizontalpacking32.c).
We improve upon and extend it to more unpacking integer types, as well as SIMD register sizes.

From the previous examples, you should now be convinced that unpacking integers requires generating many intermediary numbers, from bytes offsets to shifts and masks.
Unsurprisingly, when we switch to unpacking multiple values at a time, there are even more such numbers to compute.
Thanks to microkernel dispatching techniques, many of them can be computed ahead of time from the target unpacking integer size, the SIMD register size, and the packing bit width.
The LittleIntPacker library and previous Apache Parquet implementations both relied on a Python script that generate C++ code.
However, that prove unmanageable iterate on the algorithm, while modern C++ `constexpr` capabilities allowed us to do it directly in the source code.
At first, it seems like some constant can be computed from direct arithmetic expressions.
For instance the bit mask with `n_bits` can be computed in most cases with the formula `(1 << n_bits) - 1`.
But with increasing complexity, it becomes harder and harder to find and such formulas, as we can suspect from the previous `PackedMaxSpreadBytes` explanation.
Fortunaltely, C++ `constexpr` is a full programming language, so instead of trying to intuit expression for desired quantities, we switch to computing them at compile time!

In a sense, our compile time algorithm is the same as the final SIMD unpacking algorithm, expect that instead of running on actual data, it stores all intermediary offsets, shifts, and masks to later be used as compile-time constants in the actual unpacking.
We call this phase *plan building*, where a *plan* contains all compile time constant we wish to know about.

### Unpacking a single SIMD register

To better understand the algorithm, let us see where we start from and where we want to go.
We want to read packed values from a continuous input array, and write some unpacked values to a continuous output array.
Once we start using SIMD register, we should keep using them for all operations in order to preserve performances.
So we read some packed input values to load our say 128 bit SIMD register.
The number of values read is dependent on the packed bit size and will impact our algorithm (more on this later).
For the output we would like to write a full SIMD register with unpacked values.
Here we know easily how many unpacked values we can fit in one SIMD register.
For example with a `std::uint32_t` unpacked integer type, we can fit exactly 4 values in our 128 bit SIMD register.
Our goal is to get from one to the other, all using SIMD instructions.
So far this are the same requirements as before with a number of values hard-coded to 4.

Our SIMD register can be viewed as 4 `std::uint32_t` "slots".
If we had the relevant bits of each first four packed value in each of these slots, we could use per-slot shift and mask to get to our final register.
But that is not the case.
For instance with a packing bit size of 3, the first 4 values are all contained inside the first two bytes of our input, while the slots start at bytes 0, 4, 8, and 12.
To the rescue, we will use a byte-level swizzle (also called shuffle) that let us give a set of indices to rearange all bytes in a SIMD register (with possible copies and blanks).
With this, we can place each of our packed value somewhere in each slot.
Computing where the value will land is required to know by how much we will need to shift each slot to align the value to the begining of the slot.
This computation is very similar to what is done in the exact unpacking simulation.
And just like in it, we need to account for the byte spread when moving values accross slots.
With a packed bit size of 3, packed values can spread over 2 bytes, so we need to move 2 bytes for every packed value we want to move.
With values in each slots, and shifted to align on the begining of the slot, masking undesired bits is the easiest operation.
The mask is the same for all slots and all throughout the algorithm.

Swizzle is a relatively computationaly expensive operation (compared to other SIMD operations).
In general SIMD operations are more efficient when we do not mix the diffrent slots.
So here there is another optimization we can do with small packing sizes to reduce the number of swizzle.
We moved 2 bytes (1 value with 2 bytes spread) from the input register to each of the 4 byte slot.
In each slot, there is another 2 bytes that we filled with gibberish only to mask it out later.
That is enough to fit another value!
We can fit the next 4 values (values 4 to 8) in these bytes.
That way, to unpack these 4 values, we will only need to shift (with different shifts) and mask.
We went from 1 swizzle per 4 values unpacked to 1 swizzle per 8 values unpacked!

The following vizualization show how packed input values are extracted with SIMD with the *medium* plan we just described.

In [8]:
auto MakeSimdInput(auto const& plan) {
    auto const& shape = plan.kShape;
    auto const n_values = static_cast<std::size_t>(plan.unpacked_per_swizzle());
    
    auto const alpha_bits = MakeAlphaValues({
        .packed_bit_size = static_cast<std::size_t>(shape.packed_bit_size()),
        .n_values = n_values,
        .total_bits = static_cast<std::size_t>(shape.simd_bit_size()),
    });

    auto const palette = MakePaletteStyle(LIGHT_ORANGE, 3 * n_values / 4 + 4);
    auto packed_bit_style = MakePackedColorFn({
        .packed_bit_size = static_cast<std::size_t>(shape.packed_bit_size()),
        .n_valid_bits = n_values * static_cast<std::size_t>(shape.packed_bit_size()),
        .bit_highlight = [](auto) { return false; },
        .styles = palette,
    });

    auto styles = std::vector<TextStyle>();
    for(std::size_t k = 0; k < alpha_bits.size(); ++k){
        styles.push_back(packed_bit_style(k));
    }

    return std::pair(alpha_bits, styles);
}

auto SwizzleInput(
    auto const& plan, auto const& input, auto const& input_style,
    int read_idx = 0, int swizzle_idx = 0
) {
    auto const& shape = plan.kShape;
    auto const& swizzles = plan.swizzles.at(read_idx).at(swizzle_idx);
    auto const& all_shifts = plan.shifts.at(read_idx).at(swizzle_idx);
    
    auto swizzled = std::vector<char>(input.size(), '0');
    auto swizzled_style = std::vector<TextStyle>(input.size());
    
    for(int out_bit_idx = 0; out_bit_idx < input.size(); ++out_bit_idx){
        auto const bit_offset = out_bit_idx % kBitsPerBytes;
        auto const out_byte_idx = out_bit_idx / kBitsPerBytes;
        auto const in_byte_idx = swizzles.at(out_byte_idx);
        auto const in_bit_idx = in_byte_idx * kBitsPerBytes + bit_offset;

        // Larger indices are used to explicitly set to zero
        if (in_byte_idx >= shape.simd_byte_size()){
            continue;
        }
        swizzled.at(out_bit_idx) = input.at(in_bit_idx);
    
        auto const out_unpacked_idx = out_bit_idx / shape.unpacked_bit_size();
        auto const out_unpacked_bit_offset = out_bit_idx % shape.unpacked_bit_size();

        // Here we use the computed shifts to reverse the computation of where the value
        // needs to be colored, but in the regular algorithm , we compute where the value
        // falls in order to get the shifts.
        auto style = TextStyle{.fg = BLACK, .bg = LIGHT_GREY};
        for(auto const& shifts : all_shifts) {
            auto const shift = shifts.at(out_unpacked_idx);
            if (out_unpacked_bit_offset >= shift && out_unpacked_bit_offset < shift + shape.packed_bit_size()){
                style = input_style.at(in_bit_idx);
            }
        }
        swizzled_style.at(out_bit_idx) = style;
    }

    return std::pair(swizzled, swizzled_style);
}

auto ShiftSwizzled(
    auto const& plan, auto const& swizzled, auto const& swizzled_style,
    int read_idx = 0, int swizzle_idx = 0, int shift_idx = 0
) { 
    auto const& shape = plan.kShape;
    auto const& shifts = plan.shifts.at(read_idx).at(swizzle_idx).at(shift_idx);
    
    auto shifted = std::vector<char>();
    auto shifted_style = std::vector<TextStyle>();
    
    for(int unpacked_idx = 0; unpacked_idx < plan.unpacked_per_shifts(); ++unpacked_idx){
        auto const unpacked_bit_idx = shape.unpacked_bit_size() * unpacked_idx;
        auto const val_bit_start = unpacked_bit_idx + shifts.at(unpacked_idx); 
        auto const unpacked_bit_end = unpacked_bit_idx + shape.unpacked_bit_size();
        
        shifted.insert(shifted.end(), swizzled.cbegin() + val_bit_start, swizzled.cbegin() + unpacked_bit_end);
        shifted.resize((unpacked_idx + 1) * shape.unpacked_bit_size(), '0');
        shifted_style.insert(shifted_style.end(), swizzled_style.cbegin() + val_bit_start, swizzled_style.cbegin() + unpacked_bit_end);
        shifted_style.resize((unpacked_idx + 1) * shape.unpacked_bit_size());
    }

    return std::pair(shifted, shifted_style);
}

auto MaskRegsiter(auto const& plan, auto const& shifted, auto const& shifted_style) { 
    const auto shape = plan.kShape;
    
    auto masked = shifted;
    auto masked_style = shifted_style;
    
    for(int k = 0; k < masked.size(); ++k){
        if (k % shape.unpacked_bit_size() >= shape.packed_bit_size()){
            masked.at(k) = '0';
            masked_style.at(k) = {};
        }
    }

    return std::pair(masked, masked_style);
}

template<typename Plan>
  requires (Plan::kShape.is_medium())
auto PrintSimdExample(Plan const& plan) {
    const int read_idx = 0;
    const int swizzle_idx = 0;
    
    const auto shape = plan.kShape;

    auto const [input, input_style] = MakeSimdInput(plan); 
    std::cout << "Input register showing the " << plan.unpacked_per_swizzle()
        << " values extracted in this iteration:\n";
    PrintValuesAsBits(input, {
        .lane_bit_size = static_cast<std::size_t>(shape.unpacked_bit_size()),
        .lane_end = " ",
        .bit_style = MakeColorFn(input_style),
    }); 

    auto const [swizzled, swizzled_style] = SwizzleInput(plan, input, input_style, read_idx, swizzle_idx);  
    std::cout << "\nSwizzled register:\n";
    PrintValuesAsBits(swizzled, {
        .lane_bit_size = static_cast<std::size_t>(shape.unpacked_bit_size()),
        .lane_end = " ",
        .bit_style = MakeColorFn(swizzled_style),
    });

    for(int shift_idx = 0; shift_idx<plan.shifts.at(0).at(0).size(); ++shift_idx){
        auto const& [shifted, shifted_style] = ShiftSwizzled(plan, swizzled, swizzled_style, read_idx, swizzle_idx, shift_idx);   
        std::cout << "\n ─► Shifted register " << shift_idx << ":\n    ";
        PrintValuesAsBits(shifted, {
            .lane_bit_size = static_cast<std::size_t>(shape.unpacked_bit_size()),   
            .lane_end = " ",
            .bit_style = MakeColorFn(shifted_style),
        });
    
        auto const [masked, masked_style] = MaskRegsiter(plan, shifted, shifted_style);    
        std::cout << "\n    Masked register " << shift_idx << ":\n    ";
        PrintValuesAsBits(masked, {
            .lane_bit_size = static_cast<std::size_t>(shape.unpacked_bit_size()),
            .lane_end = " ",
            .bit_style = MakeColorFn(masked_style),
        });
    }
}

template <int kPackedBitSize, Uint kUint>
struct PrintSimdExampleDispatcher {
    /// Hard coded for visualization
    static constexpr int kSimdBitSize = 128;
    using uint_t = UintType<kUint>;
    using Traits = arrow::internal::KernelTraits<uint_t, kPackedBitSize, kSimdBitSize>;

    static void call(){
        if constexpr (Traits::kShape.is_medium()) {
            PrintSimdExample(arrow::internal::BuildMediumPlan<Traits, {32}>());
        } else {
            throw std::runtime_error("The packed bit sizes are not compatible with medium plans."); 
        }
    }
};

In [9]:
auto swizzle_wt = IntegerToolbar({
    .number_description = "Packed bit size",
    .number_min = 1,
    .uint_description = "Unpacked type",
});

auto swizzle_on_change = [](auto value, Uint uint) {   
    Dispatch<PrintSimdExampleDispatcher>(static_cast<int>(value), uint);
};

swizzle_wt.on_change(swizzle_on_change);
swizzle_wt.display();

A Jupyter widget with unique id: 765e044f9dfd44dfba89ab739ecd198e

A Jupyter widget with unique id: 96d5eebf92c84928ac48ddab57336acf

### SIMD unpacking loop

With the unpacking 3 bit values to `uint32_t` on SIMD 128 bits registers, we see that we only use little data from the input register (3 bytes out of 16).
The smaller the packed size, the more this is the case.
We therefore optimized the subsequent iterations to generate a new swizzle for the next values in the input register without seeking forward and repeating the same operation with a new input register.
We had an inner loop doing multiple shifts on a swizzled register and writing it to the output, and we are now adding an outer loop that may generate multiple swizzled register per read register.

There is one more loop we need to add, this time more as a constraint than an optimization.
If we bump the packed bit size to 17 bits from the previous example (still to `uint32_t` on SIMD 128), we notice that 4 values can get extracted leaving the next four values to start on bit index 44, which is not a multiple of 8 bits (a byte).
An implicit constraint of our micro kernel is that it can unpack values that start aligned on a byte.
In order for the next iteration to run properly, we need to leave the next values to extract to start on a byte boundary.
However, in this case, we do have not enough data left in the input register to run a second swizzle that will correct the misalignment.
Therefore, we need to add yet another outer loop, that will read the next input register.
All subsequent operation must account for a bit shift equal to the next values mis-alignement.

The following function will describe all the steps required in running a single micro kernel iteration.
Many times, some loops will only have one iterations.
In all cases, these plans are computed at compile time and this transparency enables the compiler to further optimize them (*e.g.* full loop unrolling).

In [10]:
template<typename Plan>
  requires (Plan::kShape.is_medium())
auto PrintSimdPlan(Plan const& plan) {  
    std::cout << "Medium plan for unpacking " << plan.kShape.packed_bit_size()
    << " bits to " << plan.kShape.unpacked_bit_size()
    << " bits using " << plan.kShape.simd_bit_size()
    << " bits SIMD registers:\n";
    std::cout << "   • Unpacked values per μKernel : " << plan.unpacked_per_kernel() << "\n";
    std::cout << "   • Unpacked bytes per μKernel  : " << plan.total_bytes_read() << "\n";

    std::cout << "Steps:\n";
    for (int r = 0; r < plan.reads.size(); ++r) {
        std::cout << "  ─► Read at offset: " << plan.reads.at(r) << "\n";
      
        for (int sw = 0; sw < plan.swizzles.at(r).size(); ++sw) {
            std::cout << "       ─► Swizzle with indices: ";
            PrintJoinedValues(plan.swizzles.at(r).at(sw));
            std::cout << "\n";
            
            for(int sf = 0; sf < plan.shifts.at(r).at(sw).size(); ++sf){
                std::cout << "            ─► Right shift by: ";
                PrintJoinedValues(plan.shifts.at(r).at(sw).at(sf));
                std::cout << "\n";
            }
        }
    }
}

template <int kPackedBitSize, Uint kUint>
struct PrintSimdMediumPlanDispatcher {
    /// Hard coded for visualization
    static constexpr int kSimdBitSize = 128;
    using uint_t = UintType<kUint>;
    using Traits = arrow::internal::KernelTraits<uint_t, kPackedBitSize, kSimdBitSize>;

    static void call(){
        if constexpr (Traits::kShape.is_medium()) {   
            PrintSimdPlan(arrow::internal::BuildMediumPlan<Traits, {32}>());
        } else {
            std::cerr << "The packed bit size " << kPackedBitSize << " is too large for large plans.\n";
        }
    }
};

In [11]:
auto medium_plan_wt = IntegerToolbar({
    .number_description = "Packed bit size",
    .number_min = 1,
    .uint_description = "Unpacked type",
});

auto medium_plan_on_change = [](auto value, Uint uint) {   
    Dispatch<PrintSimdMediumPlanDispatcher>(static_cast<int>(value), uint);
};

medium_plan_wt.on_change(medium_plan_on_change);
medium_plan_wt.display();

A Jupyter widget with unique id: e2ebc47a5fe24d1391624067b5d470be

A Jupyter widget with unique id: 6d108251c24d4b2281a627a12efe8e96

### Larger plan

If you tried many combinations in the prevous example, you might have found some that did not work (specifically larger packing sizes).
We have encounter this phenomenon during exact unpacking and called this phenomenon *large*.
It happens when the packing byte spread is larger than the target unpacking byte size.
In this case we dispatch to another implementation of the kernel.
We will not going into as many details, as many of the fundamentals remains the same.

The basic idea, is that during swizzle, we do not have enough bytes in each "slot" to contain a single value (due to spread).
Instead we do the next best thing.
We get as many bytes as possible swizzle, shift and mask.
Then we do it again with the remaining bytes: swizzle, shift and mask.
Finally we have two SIMD registers, each containing part of each values that we merge using bitwise OR.
One welcome simplification in this aditional complexit is that, because this happens when packing size are large, the two inner loops on shifts and swizzles always contain a single step.

The following function let you print such large plans.

In [12]:
template<typename Plan>
  requires (Plan::kShape.is_large())
auto PrintSimdPlan(Plan const& plan) {
    std::cout << "large plan for unpacking " << plan.kShape.packed_bit_size()
    << " bits to " << plan.kShape.unpacked_bit_size()
    << " bits using " << plan.kShape.simd_bit_size()
    << " bits SIMD registers:\n";
    std::cout << "   • Unpacked values per μKernel : " << plan.kUnpackedPerkernel << "\n";
    std::cout << "   • Unpacked bytes per μKernel  : " << plan.total_bytes_read() << "\n"; 

    std::cout << "Steps:\n";
    for (int r = 0; r < Plan::kReadsPerKernel; ++r) {
        std::cout << "  ─► Read at offset: " << plan.reads.at(r);
        std::cout << "\n       ─► Swizzle for low part with indices : ";
        PrintJoinedValues(plan.low_swizzles.at(r));
        std::cout << "\n       ─► Right shift for low part by : ";
        PrintJoinedValues(plan.low_rshifts.at(r));
        std::cout << "\n       ─► Swizzle for high part with indices: ";
        PrintJoinedValues(plan.high_swizzles.at(r));
        std::cout << "\n       ─► Right shift for high part by: ";
        PrintJoinedValues(plan.high_lshifts.at(r));
        std::cout << "\n       ─► Merge high and low parts with bitwise OR\n";
    }
}

template <int kPackedBitSize, Uint kUint>
struct PrintSimdLargePlanDispatcher {
    /// Hard coded for visualization
    static constexpr int kSimdBitSize = 128;
    using uint_t = UintType<kUint>;
    using Traits = arrow::internal::KernelTraits<uint_t, kPackedBitSize, kSimdBitSize>;

    static void call(){
        if constexpr (Traits::kShape.is_medium()) {   
            std::cerr << "The packed bit size " << kPackedBitSize << " is too small for large plans.\n";
        } else if constexpr (Traits::kShape.is_large()) {
            PrintSimdPlan(arrow::internal::BuildLargePlan<Traits>());
        } else if constexpr (Traits::kShape.is_oversized()) {
            std::cerr << "The packed bit size " << kPackedBitSize << " is too large for large plans.\n";
        }
    }
};

In [13]:
auto large_plan_wt = IntegerToolbar({
    .number_description = "Packed bit size",
    .number_min = 1,
    .number_default = 27,
    .uint_description = "Unpacked type",
});

auto large_plan_on_change = [](auto value, Uint uint) {   
    Dispatch<PrintSimdLargePlanDispatcher>(static_cast<int>(value), uint);
};

large_plan_wt.on_change(large_plan_on_change);
large_plan_wt.display();

A Jupyter widget with unique id: 972e134b4f2e4140ae8c74fbcceb8b25

A Jupyter widget with unique id: 29a1fdaabf924a8eadbcb919f0d42348

## Writing SIMD kernels

### Generic SIMD and fallbacks with xsimd

Now that we have computed a full plan, all that is left to do is to use them in actual SIMD code.
All the heavy lifting has been done, that is all the indices book-keeping has been done and stored in compile time arrays, ready to be used.
We only need writing our nested loop with input and output.
Or so we have left you to believe.

SIMD code can be writen using specific types and function.
However these functions vary from architecture to architecture ([x86](https://en.wikipedia.org/wiki/X86) and [ARM](https://en.wikipedia.org/wiki/ARM_architecture_family) to name the two of highest interest here).
Not only do the name of the function vary, what function are available and how they behave is different.
For Python readers, it is a bit as if [NumPy](https://numpy.org/) and [PyTorch](https://pytorch.org/) had extremely different interfaces and that you had to write an algorithm that could run with either.
What's more, x86 is old and the new functions for wider registers also do not always follow the convention of older ones.
And each generation of x86 functionality does not have a full implementation of the most basic functions (*e.g.* bitwise shift) for all numeric types.

Pretending that SIMD functionalities come as a small optimized arithmetic library is a far from the real experience.
That is what the [xsimd](https://github.com/xtensor-stack/xsimd) library, already in use in Apache Arrow, does however.
It provides a uniform interface to program SIMD algorithm independently of the targeted architecture.
More than providing nice function aliases, it implements clever fallbacks for each missing operations on each micro architectures.
Here, having computed the plan into compile time values shines again.
Fallbacks most often require extra computation and transformation of the inputs to be able to use other functions.
xsimd provides several such overloads for when some of the function inputs are known at compile time.
It this case, it can run several check to know if a simple fallack would suffice, or otherwise keep doing compile time transformations to miinimize the operations at runtime.

As part of our work, we contributed a number of such fallback improvements to xsimd.
Specifically to the `swizzle` function.
On `AVX2` (which is widely available modern `x86` CPUs), 256 bits registers are divided into two 128 bits *lanes*.
For byte-level swizzle operations, the existing CPU intruction (`_mm256_shuffle_epi8`) do not permit single bytes to cross lanes.
In other words one instructions can simultaneously do two independent permutations on 128 bits lanes (16 bytes).
Permutation indices must also be provided per-lane instead of of for the whole register.
With xsimd, knowing permutations indices at compile time, we can implement a fallback as follows.
Each line is prefix with either `[C]` if it happens at compile time or `[R]` if it happens at runtime.
The exact implementation is available in [xsimd repository](https://github.com/xtensor-stack/xsimd/blob/4d185a6de75ff51fd466064c8d91557ae459105a/include/xsimd/arch/xsimd_avx2.hpp#L1326).

```
// When Magik index is given, the resulting byte is filled with zero
[C] Define Magik = 255
[C] If indices do not require data to change lane:
        [C] Compute per lane indices taking modulo 16
        [R] Permute the input data in each lane
[C] Else if all indices point to low lane data:
        [R] Copy low lane in high lane
        [R] Permute the input data in each lane
[C] Else if all indices point to high lane data:
        [R] Copy high lane in low lane
        [C] Compute per lane indices taking modulo 16
        [R] Permute the input data in each lane
[C] Else:
        [R] Swap both lane in a second register
        [C] Compute self lane indices, replacing cross lane indices with Magik,
            taking self lane indices modulo 16.
        [C] Compute cross lane indices, replacing self lane indices with Magik,
            taking cross lane indices modulo 16.
        [R] Permute input data using self indices
        [R] Permute swapped lane data using cross lane indices
        [R] Merge input and swapped register with bitwise OR
```

This is one of the many zero-cost abstraction that shines in xsimd.
From a user perspective, it all feels like a single `xsimd::swizzle` function.
Should someone come with a better aproach, Apache Arrow will also benefit from such xsimd improvements for free.

With these three nested (compile-time) loop, we are ready to run our ready to run our *medium* micro kernel.
Our micro kernel unpacker has two major constraints.
- 1. It always extract the exact same number of values.
  Therefore we need to make sure it is not asked to extract too many or two few values.
- 2. It must extract packed values that start aligned on a byte bounday.

This is where out `unpack_exact` comes in handy.
It can be used to
- 1. Unpack remaining values onces we have run as many loops of the kernel as we can.
- 2. Unpack just the right number of values at the begining of the input array, until we find one that is properly aligned.
 
When distributing compiled packages for a wide variety of platforms, as in the case for Python and Conda pacakges, we do not know how advanced the SIMD intructions available on the client machine will be.
Instead of restricting ourselves to a minimal baseline, we compile our kernels for a set of possbile architectures.
At runtime, we must then select the micro kernel compiled for the most advanced micro architecture, and (as previously mentionned) for the proper packing bit width.
This is a small but worth upfront cost.
Using the function to extract as many values as possile in a single call will give best performances.


### Tuning
#### Integer size
Despite all fallbacks implemented in xsimd, unpacking to certain integer types does not perform as well as others.
Of course it is not fair to compare unpacking to say `uint16_t` and `uint64_t` because the latter has many more `0` to write.
However, there are two cases when we can use a different unpacking integer target to a local buffer and then copy and cast the result to the output.
- We can always decide to unpack to a larger integer size in a local buffer, (*e.g.* to `uint64_t` when the target is `uint16_t`), and then loop and cast and write to the ouput.
- When the packing width is small, we can also decide to use the same technique with a smaller integer size (*e.g.* unpack to `uint16_t` when the target is `uin64_t` is possible when the packing bit size is less than 16).

This tuning depend on CPU micro architectures and is benchmark driven.

#### Batch sizes
When unpacking 1 bit values on a 256 bits register, we can git 256 values in a kernel.
With the algorithm details here, that means that the kernel function would be called only on a number of values that is a multiple of 256 values.
In the context of of Apache Parquet this is a large.
We want the SIMD loop to start even when asked to extract a smaller number of values.
This is because the packing bit size is not static, it changes throughout the values to being packed.
We may have 512 values packed with bit size 3, followed by 200 values with bit size 10 *etc*.

We therefore adjust our planning algorithm and kernel on small packing bit sizes to limit the number of values being extracted as well as prevent out of bound reads.

## Results

## Try it out

Ignore this section for now.

In [14]:
// auto const n_values = std::size_t{1000000};
// auto const values = std::vector<std::uint32_t>(n_values, 1000);
// auto const bit_width = MaxBitWidth(values);
// auto const bytes = PackExact(values, bit_width);

In [15]:
// auto out = std::vector<std::uint32_t>(n_values);

In [16]:
// arrow::internal::unpack_128(
//     bytes.data(),
//     out.data(),
//     {
//         .batch_size = static_cast<int>(n_values),
//         .bit_width = static_cast<int>(bit_width),
//     }
// );